# SEM 畸变校正 — Jupyter 交互版

与命令行**完全等价**（同一个 `semcorr.correct_image()` API），但四张诊断图内联显示、
报告摘要直接成表。真实图像不随仓库分发，默认跑合成演示图（已知 3° 旋转 + 透视畸变）。

**用法**：改第 1 节参数 → Run All。批量处理见第 5 节。
**前置**：内核需有 `numpy / opencv-python-headless / matplotlib`（缺包时第 0 节会给出安装命令）。


## 0. 环境准备

In [ ]:
import sys, importlib.util
from pathlib import Path

# 自动定位项目根（notebook 位于项目内或其上层工作区时无需改动）
starts = []
if "__vsc_ipynb_file__" in globals():          # VS Code 注入的 notebook 路径
    starts.append(Path(globals()["__vsc_ipynb_file__"]).parent)
try:
    starts += [Path(d) for d in _dh]           # 内核工作目录
except NameError:
    pass
starts.append(Path.cwd())
ROOT = None
for start in starts:
    for cand in (start, *start.parents):
        if (cand / "src" / "semcorr").is_dir():
            ROOT = cand; break
        sub = cand / "sem-map-corrector"
        if (sub / "src" / "semcorr").is_dir():
            ROOT = sub; break
    if ROOT: break
if ROOT is not None:
    sys.path.insert(0, str(ROOT / "src"))

try:
    from semcorr import correct_image, make_demo_image
except ModuleNotFoundError as exc:
    raise RuntimeError("找不到 semcorr 包：把上方 ROOT 手动设为项目绝对路径，"
                       "或先执行 `python -m pip install -e .`") from exc

_missing = [pip for mod, pip in (("cv2", "opencv-python-headless"),
                                 ("numpy", "numpy"), ("matplotlib", "matplotlib"))
            if importlib.util.find_spec(mod) is None]
if _missing:
    raise RuntimeError("内核缺少依赖: " + ", ".join(_missing) +
                       "\n安装命令: !" + sys.executable + " -m pip install " + " ".join(_missing))

import cv2, numpy as np, matplotlib.pyplot as plt

# 中文字形：matplotlib 默认字体没有 CJK 字形，图标题里的中文会变方框。
# 在系统已装字体里挑一个中文字体（都不存在时静默退回默认）。
from matplotlib import font_manager, rcParams
_fonts = {f.name for f in font_manager.fontManager.ttflist}
for _name in ("Hiragino Sans GB", "PingFang SC", "Heiti TC", "Songti SC",
              "Arial Unicode MS", "Noto Sans CJK SC", "Microsoft YaHei", "SimHei"):
    if _name in _fonts:
        rcParams["font.sans-serif"] = [_name] + list(rcParams["font.sans-serif"])
        rcParams["axes.unicode_minus"] = False
        print("中文字体:", _name)
        break
else:
    print("中文字体: 系统无可用 CJK 字体，图标题中的中文会显示为方框")

print("semcorr 就绪 | 项目根:", ROOT, "| 内核:", sys.executable)


## 1. 参数

| 参数 | 含义 |
|---|---|
| `IMAGE_PATH` | 输入图像；`None` = 自动生成合成演示图 |
| `GRID` | 标记网格 行x列（标准版图 `2x2`；演示图 `2x2`/`2x3` 均可） |
| `OUTDIR` | 输出目录（校正图 + `diagnostics/`） |
| `AFFINE` | `False` = 分格精确单应（默认）；`True` = 全局仿射 |
| `MARK_ARM` | 校正图上红色小十字的臂长（px）；`None` = 缺省 3 px（13 像素十字） |
| `KEEP_INFO_BAR` | `False` = 自动裁掉底部 SEM 参数栏（缺省）；`True` = 保留 |

仅支持 **SE2 实心亮十字**；InLens 浮雕风格不在支持范围内。


In [ ]:
# ===== 在这里修改参数 =====
IMAGE_PATH = None                  # None = 合成演示图；正式使用改为 Path("/path/to/image.tif")
GRID = "2x2"                       # 标记网格 行x列
OUTDIR = (ROOT or Path.cwd()) / "notebook_output"
AFFINE = False                     # True = 全局仿射
MARK_ARM = None                    # 红标臂长(px)：None=3(总宽7px、13个像素)；或给绝对像素如 1
KEEP_INFO_BAR = False              # False=自动裁掉底部 SEM 参数栏；True=保留原样

if IMAGE_PATH is None:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    n, c = (int(v) for v in GRID.lower().split("x"))
    IMAGE_PATH = make_demo_image(OUTDIR / f"demo_se2_{n}x{c}.tif", n, c)
    print("演示图:", IMAGE_PATH.name, "（已知 3° 旋转 + 透视畸变）")
else:
    IMAGE_PATH = Path(IMAGE_PATH)


## 2. 运行校正

`verbose=False` 静默执行——过程细节都在报告 JSON 里，结果看下面两节。
失败（标记数不足、RMS 超限等）会直接抛 `RuntimeError`，信息里已带归因：
列出已接受标记的模板/对称/形状分，指出缺失的是哪个格位、它应当在哪、以及
该位置逐项的门槛判定，并在 `diagnostics/` 下留一张 `*_detection_failed.png`
（绿=接受 红=剔除 橙=缺失格位预测位置），放大橙色圈即可确认十字是否残缺。


In [ ]:
report = correct_image(IMAGE_PATH, grid=GRID, outdir=str(OUTDIR),
                       affine=AFFINE, mark_arm=MARK_ARM,
                       keep_info_bar=KEEP_INFO_BAR, verbose=False)
print("状态:", report["quality_status"], "| 模型:", report["method"],
      "| 拟合 RMS: %.3f px" % report["used_model_rms_px"])


## 3. 结果图

- **检测诊断**：绿圈 = 接受的候选，红 × = 拒绝的干扰物
- **残差诊断**：箭头 = 残差方向（×20），橙圈 = 留一验证可疑点
- **校正图**：mark 中心严格构成正方形网格（分格精确单应），并就地用红色
  小十字标出每个中心（只做定位指示，不写坐标；数值见
  `diagnostics/*_centers_corrected.csv`）
- **原图中心标注**：原图坐标系下的中心（`diagnostics/*_centers.csv`）


In [ ]:
outs = report["outputs"]

def _load(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 and img.shape[2] == 3 else img

panels = [(outs["detection_overlay"], "检测诊断（绿=接受，红=拒绝）"),
          (outs["residual_plot"],     "残差诊断（箭头 ×20，橙圈=留一可疑）"),
          (outs["corrected_image"],   "校正图（红小十字 = mark 中心）"),
          (outs["centers_annotated"], "原图中心标注（原图坐标系）")]

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
for ax, (path, title) in zip(axes.ravel(), panels):
    ax.imshow(_load(path), cmap="gray")
    ax.set_title(title, fontsize=13)
    ax.axis("off")
plt.tight_layout(); plt.show()


## 4. 报告摘要

完整机器可读报告（仿射/单应参数、自愈日志、留一验证、SHA256）在
`OUTDIR/diagnostics/<图像名>_report.json`。


In [ ]:
sc = report["self_check"]
warn = "（" + "；".join(report["quality_warnings"]) + "）" if report["quality_warnings"] else ""
print("质量:", report["quality_status"], warn)
print("理想坐标:", report["ideal_source"])
print("校正后自检: %d/%d 标记复检，RMS = %s px" %
      (sc["n_detected"], sc["n_total"],
       "—" if sc["rms"] is None else "%.3f" % sc["rms"]))
if report["repair"]:
    print("网格自愈:", ", ".join(r["slot"] for r in report["repair"]))

print("\n%-4s %10s %10s %10s" % ("mark", "检测x", "检测y", "残差(px)"))
for m in report["marks"]:
    r = (m["residual_px"][0] ** 2 + m["residual_px"][1] ** 2) ** 0.5
    tag = "  [网格恢复]" if m["recovered_from_grid_search"] else ""
    print("%-4s %10.3f %10.3f %10.3f%s"
          % (m["id"], m["detected_px"][0], m["detected_px"][1], r, tag))

cc = report["corrected_centers"]
print("\n校正后中心坐标（%s，见 *_corrected.tif 内嵌的红色小十字）：" % cc["unit"])
for p in cc["points"]:
    tag = "" if p["source"] == "self-check" else "  [未复检，取理想格位]"
    print("%-4s %10.3f %10.3f%s" % (p["id"], p["x"], p["y"], tag))


## 5. 批量处理（可选）

等价于 `./semcorr --batch <目录>`。把 `BATCH_DIR` 设为图像文件夹即可。


In [ ]:
BATCH_DIR = None   # 例如 Path("/path/to/SEM_images")；None = 跳过本节

if BATCH_DIR is None:
    print("跳过批量处理（把 BATCH_DIR 设为图像文件夹即可启用）")
else:
    from semcorr.io import list_images
    files = list_images(BATCH_DIR)
    batch_out = Path(BATCH_DIR) / "corrected"
    print(f"批量处理 {len(files)} 张 → {batch_out}\n")
    rows = []
    for i, p in enumerate(files, 1):
        try:
            rep = correct_image(p, grid=GRID, outdir=str(batch_out),
                                affine=AFFINE, mark_arm=MARK_ARM,
                                keep_info_bar=KEEP_INFO_BAR,
                                verbose=False)
            rows.append((p.name, rep["quality_status"], rep["used_model_rms_px"],
                         rep["self_check"]["n_detected"], rep["self_check"]["n_total"]))
        except (RuntimeError, FileNotFoundError, ValueError) as exc:
            rows.append((p.name, "FAIL: " + str(exc)[:60], None, None, None))
        print("[%d/%d] %s" % (i, len(files), rows[-1][0]), rows[-1][1])
    print("\n%-28s %-14s %9s %7s" % ("图像", "状态", "RMS(px)", "自检"))
    for name, status, rms, nd, nt in rows:
        print("%-28s %-14s %9s %7s" %
              (name[:28], status, "—" if rms is None else "%.3f" % rms,
               "—" if nd is None else "%d/%d" % (nd, nt)))


## 结果解读（正式实验前必读）

- **先看检测诊断图**：确认绿圈落在十字中心——JSON 残差为零/很小只说明模型穿过输入点，不单独证明中心正确。
- **橙圈 = 留一可疑点**：剔除后 RMS 显著下降者疑似离群，需人工核查。
- **校正后自检 RMS** 应处于单点检测噪声量级；明显偏大说明校正有问题。
- `WARN_REVIEW` = 有警告需人工复核（如部分标记未检出）；`PASS` = 全部通过。
- 输入图像**不会被修改**；所有输出写入 `OUTDIR`。
